# Feature Engineering and Nonlinearity

Goal: Add polynomial features to capture nonlinearity in medical data.

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer




def load_dataset(name: str) -> pd.DataFrame:
    base_url = (
        "https://raw.githubusercontent.com/"
        "Jesse-Islam/QLS-MiCM_Introduction_to_supervised_machine_learning/main/Exercises/data/"
    )
    url = f"{base_url}{name}.csv"
    
    try:
        return pd.read_csv(url)
    except Exception as e:
        raise FileNotFoundError(f"Could not load dataset '{name}' from {url}") from e



def basic_train_test_split(df: pd.DataFrame, target: str, test_size: float = 0.2, random_state: int = 42):
    X = df.drop(columns=[target])
    y = df[target]
    return train_test_split(X, y, test_size=test_size, random_state=random_state)


def build_preprocessor(num_features, cat_features):
    numeric_transformer = StandardScaler()
    categorical_transformer = OneHotEncoder(handle_unknown="ignore")
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, num_features),
            ("cat", categorical_transformer, cat_features),
        ]
    )
    return preprocessor


from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ==============================
# Load dataset and train models
# ==============================

df = load_dataset("drug_response")
# Basic summary
display(df.head())
print(f"Shape: {df.shape}")
print(df.describe().T)

# Check for missing data
print("\nMissing values per column:")
print(df.isnull().sum())


# ==============================
# Visual exploration
# ==============================

# Correlation overview
corr = df.corr(numeric_only=True)
print("\nTop features correlated with 'effect':")
print(corr["effect"].drop("effect").sort_values(ascending=False).head(5))

# 1. Target distribution
plt.figure(figsize=(6,4))
plt.hist(df["effect"], bins=25, color="skyblue", edgecolor="black")
plt.title("Distribution of drug effect")
plt.xlabel("effect")
plt.ylabel("count")
plt.grid(alpha=0.3)
plt.show()

# 2. Relationship with the strongest predictor
# Find the feature most correlated with the target
corrs = df.corr(numeric_only=True)["effect"].drop("effect").sort_values(ascending=False)
top_feature = corrs.index[0]
print(f"Top correlated feature: {top_feature} (corr = {corrs.iloc[0]:.2f})")

plt.figure(figsize=(6,4))
plt.scatter(df[top_feature], df["effect"], alpha=0.7, edgecolor="black")
plt.title(f"Drug effect vs {top_feature}")
plt.xlabel(top_feature)
plt.ylabel("effect")
plt.grid(alpha=0.3)
plt.show()


In [ ]:

X_train, X_test, y_train, y_test = basic_train_test_split(df, target="effect")

# Baseline linear regression
base_model = LinearRegression()
base_model.fit(X_train, y_train)
base_pred = base_model.predict(X_test)
print("Baseline MAE:", mean_absolute_error(y_test, base_pred))

# Polynomial model
degree = 3
poly_model = Pipeline(
    steps=[
        ("poly", PolynomialFeatures(degree=degree, include_bias=False)),
        ("model", LinearRegression()),
    ]
)
poly_model.fit(X_train, y_train)
poly_pred = poly_model.predict(X_test)
print(f"Polynomial (degree={degree}) MAE:", mean_absolute_error(y_test, poly_pred))


# 3. Visualize model fit (using the top feature only)
# We'll fit both linear and polynomial models in 1D to visualize curvature
x_col = top_feature
X_single = X_train[[x_col]]
y_single = y_train

# Fit linear and polynomial models on single predictor
lin = LinearRegression().fit(X_single, y_single)
poly = Pipeline(
    steps=[
        ("poly", PolynomialFeatures(degree=degree, include_bias=True)),
        ("model", LinearRegression()),
    ]
).fit(X_single, y_single)

# Generate smooth prediction line
x_range = np.linspace(X_single.min().values[0], X_single.max().values[0], 200)
x_range_df = pd.DataFrame({x_col: x_range})

y_lin_pred = lin.predict(x_range_df)
y_poly_pred = poly.predict(x_range_df)

# Plot predictions
plt.figure(figsize=(7,5))
plt.scatter(X_single, y_single, alpha=0.5, color="gray", label="training data")
plt.plot(x_range, y_lin_pred, color="blue", label="linear fit")
plt.plot(x_range, y_poly_pred, color="red", label=f"polynomial (deg={degree}) fit")
plt.title(f"Model fits for {x_col}")
plt.xlabel(x_col)
plt.ylabel("effect")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


As the degree of the polynomial gets bigger, does fit get better?

is there a way to keep things under control even when degree is too high?